# 🧩 Thí nghiệm Tiling/SAHI – Cải thiện phát hiện rác nhỏ (Stage 1)

**Mục tiêu:** Kiểm tra xem việc cắt ảnh lớn thành các tile nhỏ (Tiling/SAHI) có giúp mô hình YOLOv8s tăng chỉ số Recall (đặc biệt là phát hiện các vật thể rác rất nhỏ như đầu lọc thuốc lá, nắp chai) hay không.

**Tại sao cần thiết?**
Dataset TACO chứa nhiều ảnh có độ phân giải siêu cao (vd: 3000x4000). Khi đưa vào YOLOv8 với `imgsz=640`, ảnh bị thu nhỏ rất mạnh, khiến những cục rác nhỏ bị "bóp nát" chỉ còn vài pixel và biến mất. Kỹ thuật Tiling cắt ảnh 3000x4000 thành nhiều mảnh 640x640 (giữ nguyên độ phân giải), giúp YOLO dễ dàng nhìn thấy rác nhỏ.

⚠️ **Yêu cầu:** GPU accelerator + Internet enabled

In [ ]:
# ============================================================
# Cell 1: Clone repo + Cài đặt thư viện
# ============================================================
!git clone https://github.com/Shiba-dotcom/waste-detection2-Stage.git
!pip install -q ultralytics

In [ ]:
# ============================================================
# Cell 2: Chuẩn bị dữ liệu (Copy Data)
# ============================================================
import os, shutil

!mkdir -p /kaggle/working/waste-detection2-Stage/data/external/TrashNet
!mkdir -p /kaggle/working/waste-detection2-Stage/data/external/RealWaste
!mkdir -p /kaggle/working/waste-detection2-Stage/data/raw

datasets_to_copy = [
    {"src": "/kaggle/input/datasets/feyzazkefe/trashnet/dataset-resized",
     "dst": "/kaggle/working/waste-detection2-Stage/data/external/TrashNet"},
    {"src": "/kaggle/input/datasets/sohamchaudhari2004/taco-trash-detection-dataset/data",
     "dst": "/kaggle/working/waste-detection2-Stage/data/raw"},
    {"src": "/kaggle/input/datasets/joebeachcapital/realwaste/realwaste-main/RealWaste",
     "dst": "/kaggle/working/waste-detection2-Stage/data/external/RealWaste"}
]

for task in datasets_to_copy:
    if os.path.exists(task["src"]):
        os.makedirs(task["dst"], exist_ok=True)
        shutil.copytree(task["src"], task["dst"], dirs_exist_ok=True)
        print(f"OK: {os.path.basename(task['src'])}")
    else:
        print(f"SKIP: {task['src']}")

In [ ]:
# ============================================================
# Cell 3: Tiền xử lý dữ liệu & CHẠY TILING
# ============================================================
%cd /kaggle/working/waste-detection2-Stage
!python src/data_prep/data_cleaning.py
!python src/Training_dataYolo.py
!python src/data_prep/split_dataset.py

# CHẠY SCRIPT TILING
# Bước này sẽ sinh ra thư mục: data/processed_tiled
!python src/data_prep/tiling.py
print("\n✅ Dữ liệu Tiled đã sẵn sàng!")

In [ ]:
# ============================================================
# Cell 4: Tạo file data.yaml cho tập Tiled nếu chưa có
# ============================================================
import yaml
from pathlib import Path

TILED_DIR = Path("/kaggle/working/waste-detection2-Stage/data/processed_tiled")
YAML_PATH = TILED_DIR / "data.yaml"

data_yaml = {
    "path": str(TILED_DIR.resolve()),
    "train": "images/train",
    "val": "images/val",
    "test": "images/test",
    "names": {0: "Waste"}
}

with open(YAML_PATH, "w") as f:
    yaml.dump(data_yaml, f, sort_keys=False)

print(f"[INFO] Đã tạo file yaml: {YAML_PATH}")

In [ ]:
# ============================================================
# Cell 5: Train YOLOv8s trên dữ liệu Tiled
# ============================================================
from ultralytics import YOLO
import time

DATASET_YAML = "/kaggle/working/waste-detection2-Stage/data/processed_tiled/data.yaml"
OUTPUT_DIR   = "/kaggle/working/waste-detection2-Stage/results/yolov8s_tiled_runs"

print("=" * 60)
print("  🚀 Bắt đầu train YOLOv8s (Tiled/SAHI)")
print("=" * 60)

model_s = YOLO("yolov8s.pt")

t_start = time.time()

results_train = model_s.train(
    data        = DATASET_YAML,
    imgsz       = 640,
    epochs      = 100,
    batch       = 16,
    patience    = 20,
    optimizer   = "auto",
    lr0         = 0.01,
    cos_lr      = True,
    augment     = True,
    workers     = 4,
    project     = OUTPUT_DIR,
    name        = "stage1_yolov8s_tiled",
    exist_ok    = True,
    save        = True,
    save_period = -1,
    plots       = True,
    verbose     = True,
)

train_time = time.time() - t_start
print(f"\n✅ Train hoàn tất! Thời gian: {train_time/60:.1f} phút")

In [ ]:
# ============================================================
# Cell 6: Đánh giá nhanh bằng model.val()
# ============================================================
from ultralytics import YOLO

BEST_WEIGHTS = "/kaggle/working/waste-detection2-Stage/results/yolov8s_tiled_runs/stage1_yolov8s_tiled/weights/best.pt"
model_s = YOLO(BEST_WEIGHTS)

print("=" * 60)
print("  YOLOv8s Tiled – Validation (Chốt cứng conf = 0.15)")
print("=" * 60)
val_015 = model_s.val(
    data=DATASET_YAML, imgsz=640, batch=16,
    split="val", conf=0.15, verbose=True
)

In [ ]:
# ============================================================
# Cell 7: Download kết quả
# ============================================================
import shutil
shutil.make_archive("/kaggle/working/yolov8s_tiled_results", "zip",
                    "/kaggle/working/waste-detection2-Stage/results/yolov8s_tiled_runs")
print("\n📦 Kết quả đã nén: /kaggle/working/yolov8s_tiled_results.zip")